# Notebook 1: Technical Indicator Strategy

**Purpose**: Generate BUY/SELL/HOLD signals using weighted voting across EMA crossover, RSI, Bollinger Bands, and OBV. Backtest and save results.

**Prerequisites**: Run `0_data_and_features.ipynb` first.

**GPU needed**: No (CPU only, runs in ~1 min)

In [1]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')

from config import (
    ALL_TICKERS, FEATURES_DIR, CAPITAL_PER_ASSET,
    INDICATOR_WEIGHTS, BUY_THRESHOLD, SELL_THRESHOLD,
    RSI_PERIOD,
)
from utils import (
    setup_logging, create_directories, load_exchange_rates,
    simulate_trading, compute_metrics, compute_buy_and_hold,
    save_results, is_indian_stock,
)

logger = setup_logging('technical_strategy')
create_directories()

# load common start date
with open(os.path.join('data', 'common_start_date.json'), 'r') as f:
    common_start = json.load(f)['common_start_date']
print(f'Common backtest start: {common_start}')

# load exchange rates for INR conversion
exchange_rates = load_exchange_rates()
print('Setup complete.')

Common backtest start: 2022-04-30
Setup complete.


## 1. Load Feature Data

In [2]:
def load_features(ticker):
    """Load pre-computed features for a single ticker."""
    safe_name = ticker.replace('=', '_').replace('-', '_')
    path = os.path.join(FEATURES_DIR, f'{safe_name}_features.parquet')
    if not os.path.exists(path):
        raise FileNotFoundError(f'Features not found for {ticker}. Run notebook 0 first.')
    return pd.read_parquet(path)

# load all
feature_data = {}
for ticker in ALL_TICKERS:
    feature_data[ticker] = load_features(ticker)
print(f'Loaded features for {len(feature_data)} assets.')

Loaded features for 20 assets.


## 2. Signal Generation — Weighted Voting

Each indicator votes: +1 (BUY), -1 (SELL), or 0 (HOLD).
Weighted sum is compared against thresholds.

| Indicator | Weight | BUY condition | SELL condition |
|-----------|--------|---------------|----------------|
| EMA 20/50 | 2.0 | EMA20 > EMA50 | EMA20 < EMA50 |
| RSI(14) | 1.5 | RSI < 30 (oversold) | RSI > 70 (overbought) |
| Bollinger | 1.0 | Close < Lower band | Close > Upper band |
| OBV | 1.0 | OBV > OBV_EMA20 | OBV < OBV_EMA20 |

In [3]:
def generate_technical_signals(df):
    """
    Generate BUY/SELL/HOLD signals using weighted indicator voting.
    Signals are shifted by 1 day (signal on day T → trade on day T+1).
    
    Returns: Series with 'BUY'/'SELL'/'HOLD' values, same index as df.
    """
    w = INDICATOR_WEIGHTS
    
    # EMA crossover: bullish when short EMA above long EMA
    ema_vote = np.where(
        df['ema_20'] > df['ema_50'], 1,
        np.where(df['ema_20'] < df['ema_50'], -1, 0)
    )
    
    # RSI: oversold = buy opportunity, overbought = sell
    rsi_vote = np.where(
        df['rsi_14'] < 30, 1,
        np.where(df['rsi_14'] > 70, -1, 0)
    )
    
    # Bollinger Bands: price below lower = buy, above upper = sell
    bb_vote = np.where(
        df['Close'] < df['bb_lower'], 1,
        np.where(df['Close'] > df['bb_upper'], -1, 0)
    )
    
    # OBV: rising OBV (above its EMA) confirms buying pressure
    obv_vote = np.where(
        df['obv'] > df['obv_ema_20'], 1,
        np.where(df['obv'] < df['obv_ema_20'], -1, 0)
    )
    
    # weighted composite score
    score = (
        w['ema_crossover'] * ema_vote +
        w['rsi'] * rsi_vote +
        w['bollinger'] * bb_vote +
        w['obv'] * obv_vote
    )
    
    # threshold to signals
    raw_signal = np.where(
        score >= BUY_THRESHOLD, 'BUY',
        np.where(score <= SELL_THRESHOLD, 'SELL', 'HOLD')
    )
    
    # shift by 1 day: signal generated at close of T, trade at open of T+1
    signals = pd.Series(raw_signal, index=df.index).shift(1).fillna('HOLD')
    
    return signals

## 3. Run Strategy on All Assets

In [4]:
portfolio_histories = {}
order_books = {}
metrics = {}
bh_metrics = {}  # buy-and-hold baseline

for i, (ticker, df) in enumerate(feature_data.items()):
    print(f'[{i+1}/{len(feature_data)}] {ticker}...', end=' ')
    
    # generate signals (computed over full period)
    signals = generate_technical_signals(df)
    
    # backtest (trades only from common start date)
    exr = exchange_rates if is_indian_stock(ticker) else None
    ph, ob = simulate_trading(
        df, signals, ticker,
        initial_capital=CAPITAL_PER_ASSET,
        exchange_rates=exr,
        start_date=common_start,
    )
    
    portfolio_histories[ticker] = ph
    order_books[ticker] = ob
    
    # compute metrics
    m = compute_metrics(ph)
    metrics[ticker] = m
    
    # buy-and-hold baseline
    bh = compute_buy_and_hold(df, ticker, CAPITAL_PER_ASSET, exr, common_start)
    bh_m = compute_metrics(bh)
    bh_metrics[ticker] = bh_m
    
    print(f'Return: {m["total_return"]*100:+.1f}% | Sharpe: {m["sharpe_ratio"]:.2f} | '
          f'MaxDD: {m["max_drawdown"]*100:.1f}% | Trades: {m["num_trades"]} | '
          f'B&H: {bh_m["total_return"]*100:+.1f}%')

print('\nAll assets processed.')

[1/20] AAPL... Return: +17.7% | Sharpe: 0.69 | MaxDD: 11.7% | Trades: 467 | B&H: +61.7%
[2/20] MSFT... Return: +11.0% | Sharpe: 0.40 | MaxDD: 12.0% | Trades: 372 | B&H: +54.8%
[3/20] GOOGL... Return: +16.3% | Sharpe: 0.52 | MaxDD: 7.0% | Trades: 518 | B&H: +67.9%
[4/20] AMZN... Return: +20.3% | Sharpe: 0.74 | MaxDD: 6.4% | Trades: 525 | B&H: +77.6%
[5/20] NVDA... Return: +53.9% | Sharpe: 1.03 | MaxDD: 10.9% | Trades: 430 | B&H: +642.2%
[6/20] META... Return: +46.7% | Sharpe: 1.11 | MaxDD: 10.8% | Trades: 354 | B&H: +195.0%
[7/20] TSLA... Return: +10.1% | Sharpe: 0.34 | MaxDD: 16.6% | Trades: 378 | B&H: +43.2%
[8/20] JPM... Return: +23.8% | Sharpe: 0.79 | MaxDD: 9.9% | Trades: 463 | B&H: +114.3%
[9/20] JNJ... Return: -12.2% | Sharpe: -0.60 | MaxDD: 14.2% | Trades: 417 | B&H: -13.9%
[10/20] V... Return: +19.9% | Sharpe: 0.72 | MaxDD: 6.2% | Trades: 416 | B&H: +50.7%
[11/20] RELIANCE.NS... Return: +5.9% | Sharpe: 0.21 | MaxDD: 14.5% | Trades: 432 | B&H: -14.8%
[12/20] TCS.NS... Return: +9

## 4. Results Summary

In [5]:
# build comparison table
results_table = []
for ticker in metrics:
    m = metrics[ticker]
    bh = bh_metrics[ticker]
    results_table.append({
        'Ticker': ticker,
        'Strategy Return': f"{m['total_return']*100:+.2f}%",
        'B&H Return': f"{bh['total_return']*100:+.2f}%",
        'Sharpe': f"{m['sharpe_ratio']:.2f}",
        'Max Drawdown': f"{m['max_drawdown']*100:.1f}%",
        'Trades': m['num_trades'],
        'Alpha': f"{(m['total_return'] - bh['total_return'])*100:+.2f}%",
    })

results_df = pd.DataFrame(results_table)
results_df

,Ticker,Strategy Return,B&H Return,Sharpe,Max Drawdown,Trades,Alpha
0,AAPL,+17.69%,+61.67%,0.69,11.7%,467,-43.98%
1,MSFT,+10.99%,+54.83%,0.40,12.0%,372,-43.84%
2,GOOGL,+16.28%,+67.92%,0.52,7.0%,518,-51.64%
3,AMZN,+20.29%,+77.61%,0.74,6.4%,525,-57.32%
4,NVDA,+53.88%,+642.23%,1.03,10.9%,430,-588.35%
5,META,+46.67%,+194.98%,1.11,10.8%,354,-148.31%
6,TSLA,+10.05%,+43.23%,0.34,16.6%,378,-33.18%
7,JPM,+23.77%,+114.31%,0.79,9.9%,463,-90.54%
8,JNJ,-12.15%,-13.85%,-0.60,14.2%,417,+1.70%
9,V,+19.85%,+50.75%,0.72,6.2%,416,-30.90%


## 5. Save Results

In [6]:
# save portfolio histories and order books
save_results(portfolio_histories, order_books, 'technical')

# also save buy-and-hold results for comparison notebook
bh_histories = {}
for ticker, df in feature_data.items():
    exr = exchange_rates if is_indian_stock(ticker) else None
    bh_histories[ticker] = compute_buy_and_hold(df, ticker, CAPITAL_PER_ASSET, exr, common_start)
save_results(bh_histories, {}, 'buy_and_hold')

# save metrics as JSON for quick access
import json
metrics_path = os.path.join('results', 'technical_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print('Results saved to results/ directory.')
print('Notebook 1 complete.')

Results saved to results/ directory.
Notebook 1 complete.
